In [ ]:
]activate ../../../

In [ ]:
using Revise
includet("./base.jl")
using LinearAlgebra
using Printf

In [ ]:
using CairoMakie
CairoMakie.activate!()

# Data

In [ ]:
f = jldopen("./gd1_260731_115325.jld2")
N = f["N"]; M = f["M"]; DN = f["DN"]; DR = f["DR"]
Ks = f["Ks"]; lis = f["lis"]; lsks = f["lsks"]
rdfs = f["raw_dfs"]
@show N M DN DR length(Ks) lis [nrow(d) for d in rdfs];

# Peak eigenvector vs extinct strains

Per system with a positive peak: find the peak of the dispersion relation over `lsks`, take the
eigenvector of the leading eigenvalue there, normalize by its largest component and record the
largest component sitting on a strain with `|N_i| < extinctthr` in the homogeneous steady state.

`invl_i = M1[i, i]` is the eigenvalue of an extinct strain's own row (it decouples when `N_i = 0`),
ie its invasion growth rate. `mrl_range` is max - min of the mrl over the whole k range.

In [ ]:
const EXTINCTTHR = 1e-10

function peak_eigvec_info(sp, ss, ks; extinctthr=EXTINCTTHR)
    Ns, Nr = get_Ns(sp)
    M1 = make_M1(sp, ss)
    Ds = get_Ds(sp)
    Mw = similar(M1)

    mrls = Vector{Float64}(undef, length(ks))
    for (ki, k) in enumerate(ks)
        Mw .= M1
        M1_to_M!(Mw, Ds, k)
        mrls[ki] = maximum(real, eigvals!(Mw))
    end
    maxmrl, maxi = findmax(mrls)

    Mw .= M1
    M1_to_M!(Mw, Ds, ks[maxi])
    E = eigen!(Mw; sortby=eigen_sortby_reverse)
    av = abs.(E.vectors[:, 1])
    av ./= maximum(av)

    ext = [abs(ss[i]) < extinctthr for i in 1:Ns]
    invls = [M1[i, i] for i in 1:Ns]
    nse = count(ext)
    major = av[1:Ns] .> 0.1

    (;
        maxmrl, maxi, kpeak=ks[maxi],
        lam_re=real(E.values[1]), lam_im=imag(E.values[1]),
        num_extinct=nse,
        rel_extinct=(nse > 0 ? maximum(av[1:Ns][ext]) : 0.0),
        which_extinct=(nse > 0 ? ((1:Ns)[ext])[argmax(av[1:Ns][ext])] : 0),
        rel_alive=(nse < Ns ? maximum(av[1:Ns][.!ext]) : 0.0),
        rel_res=maximum(av[Ns+1:end]),
        max_invl_extinct=(nse > 0 ? maximum(invls[ext]) : -Inf),
        min_ss_major=(any(major) ? minimum(ss[1:Ns][major]) : NaN),
        mrl_range=maximum(mrls) - minimum(mrls),
    )
end

Scan all systems that have a positive peak (~10 min with a few threads).

In [ ]:
function scan_all(rdfs, lis, Ks, ks; peakthr=1000 * eps())
    res = DataFrame(;
        lii=Int[], li=Float64[], K=Float64[], ri=Int[],
        sscode=Int[], lscode=Int[], k0mrl=Float64[],
        maxmrl=Float64[], maxi=Int[], kpeak=Float64[],
        lam_re=Float64[], lam_im=Float64[],
        num_extinct=Int[], rel_extinct=Float64[], which_extinct=Int[],
        rel_alive=Float64[], rel_res=Float64[], max_invl_extinct=Float64[],
        min_ss_major=Float64[], mrl_range=Float64[],
    )
    lk = ReentrantLock()
    pb = Progress(sum(nrow(d) for d in rdfs))
    for (j, df) in enumerate(rdfs)
        Threads.@threads for i in 1:nrow(df)
            r = df[i, :]
            if (r.sscode in (1, 2)) && !ismissing(r.maxmrl) && (r.maxmrl > peakthr)
                info = peak_eigvec_info(r.params, r.steadystates, ks)
                lock(lk) do
                    push!(res, (; lii=j, li=lis[j], K=r.K, ri=i,
                        sscode=r.sscode, lscode=r.lscode, k0mrl=r.k0mrl, info...))
                end
            end
            next!(pb)
        end
    end
    finish!(pb)
    res
end

res = scan_all(rdfs, lis, Ks, lsks);
nrow(res)

# Filtering for extinct strains in the peak mode

In [ ]:
relthr = 1e-3
hits = @subset res :rel_extinct .> relthr
@show nrow(hits) maximum(res.rel_extinct)
hits

In [ ]:
# for the hits - k0mrl, the peak, the extinct strains' invasion rates and how flat the
# dispersion relation is over the whole k range
first(sort(hits, :maxmrl; rev=true), 20)[:, [:li, :K, :lscode, :k0mrl, :maxmrl, :max_invl_extinct, :mrl_range, :maxi, :kpeak, :num_extinct, :rel_extinct]]

In [ ]:
# smallest steady state biomass of a strain that carries >0.1 of the peak eigenvector
clean = @subset res :lscode .== 2
extrema(filter(!isnan, clean.min_ss_major)), maximum(clean.rel_extinct)

In [ ]:
# how the largest extinct-strain component is distributed
fig = Figure()
ax = Axis(fig[1, 1]; xlabel="log10(largest extinct-strain component of peak eigenvector)")
hist!(ax, log10.(res.rel_extinct[res.rel_extinct.>0]); bins=60)
vlines!(ax, log10(relthr); color=:red)
fig

In [ ]:
# same split by lscode
combine(groupby(res, :lscode),
    nrow,
    :rel_extinct => maximum,
    :max_invl_extinct => maximum,
    [:lam_re, :max_invl_extinct] => ((l, i) -> minimum(l .- i)) => :min_gap_to_peak,
)

# Plotting individual cases

In [ ]:
function plot_case(row; ks=lsks, extinctthr=EXTINCTTHR, ylims=nothing)
    r = rdfs[row.lii][row.ri, :]
    sp, ss = r.params, r.steadystates
    Ns, Nr = get_Ns(sp)
    M1 = make_M1(sp, ss)
    ext = [abs(ss[i]) < extinctthr for i in 1:Ns]

    lams = linstab_make_k_func(sp, ss; returnobj=:evals).(ks)

    Mw = M1_to_M(M1, get_Ds(sp), row.kpeak)
    E = eigen!(Mw; sortby=eigen_sortby_reverse)
    av = abs.(E.vectors[:, 1])
    av ./= maximum(av)
    av = max.(av, 1e-20)

    fig = Figure(; size=(1000, 380))

    ax1 = Axis(fig[1, 1]; xscale=log10, xlabel="k", ylabel="Re(λ)",
        title=(@sprintf "li=%g K=%.3g sscode=%d lscode=%d" row.li row.K row.sscode row.lscode))
    for i in 1:(Ns+Nr)
        lines!(ax1, ks, real(getindex.(lams, i)); color=(:black, 0.3))
    end
    for i in (1:Ns)[ext]
        hlines!(ax1, M1[i, i]; color=(:dodgerblue, 0.6), linestyle=:dash)
    end
    hlines!(ax1, 0.0; color=(:black, 0.3), linestyle=:dot)
    scatter!(ax1, [row.kpeak], [row.maxmrl]; color=:red)
    if isnothing(ylims)
        ylims!(ax1, -3 * abs(row.maxmrl), 2 * abs(row.maxmrl))
    else
        ylims!(ax1, ylims...)
    end

    ax2 = Axis(fig[1, 2]; yscale=log10, xlabel="component",
        ylabel="|v| / max|v| at the peak",
        title=(@sprintf "rel_extinct=%.3g, strain %d" row.rel_extinct row.which_extinct))
    scatter!(ax2, (1:Ns)[.!ext], av[1:Ns][.!ext]; color=:black, label="alive strains")
    scatter!(ax2, (1:Ns)[ext], av[1:Ns][ext]; color=:dodgerblue, label="extinct strains")
    scatter!(ax2, (Ns+1):(Ns+Nr), av[Ns+1:end]; color=:grey, marker=:rect, label="resources")
    axislegend(ax2; position=:rb)

    fig
end

In [ ]:
top = first(sort(res, :rel_extinct; rev=true), 5)

In [ ]:
plot_case(top[1, :])

In [ ]:
plot_case(top[2, :])

# Extinct strains closest to invading

`max_invl_extinct` is the largest invasion growth rate among the extinct strains. With `DN = 0`
these branches are flat in k, so they reach the peak only when `k0mrl >= 0`.

In [ ]:
sort(res, :max_invl_extinct; rev=true)[1:10, [:li, :K, :sscode, :lscode, :k0mrl, :maxmrl, :kpeak, :lam_re, :max_invl_extinct, :rel_extinct, :num_extinct]]

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1]; xlabel="max invasion growth rate of an extinct strain", ylabel="peak λ")
scatter!(ax, res.max_invl_extinct, res.lam_re; markersize=4, color=(:black, 0.2))
ablines!(ax, 0.0, 1.0; color=:red)
fig